# Tool Calling
# 0. 介绍

**研究背景**：Agent 不只要生成文字，还要调用搜索、数据库或文件操作等工具，才能读取外部信息并改变真实环境。为了让大模型能够使用这些能力，外层程序必须用机器可以理解的方式说明有哪些工具、每个工具需要什么参数，以及调用后返回什么结果。

**现存问题**：大模型生成的内容具有不确定性，工具名称和参数可能出现格式错误、字段缺失或类型不符。如果外层程序把这些内容直接交给工具，轻则调用失败，重则把错误数据写入真实系统；仅凭模型说“任务已完成”，也无法确认工具是否真的执行成功。

**解决方案**：本 Notebook 将实现一个极简的 Tool Calling，采用当前学术界和工业界通用的`函数调用 + JSON Schema + 执行前校验`机制：先向大模型提供结构化工具定义，再把模型返回的工具请求解析并校验，只有合法参数才能进入工具，最后用调用 ID 关联请求与结果。然后用同一份真实 API 操作进行对比：基线版本直接执行未校验参数而产生错误结果，改进版本在执行前拦截无效请求并完成合法调用，从而直观看到工具协议如何把不确定的模型输出变成可验证的程序操作。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以调用的工具、需要完成的任务，以及判断工具结果是否正确的标准。

# 2. 前置准备
## 2.1 定义库存工具
大模型只知道训练时学到的内容，不知道程序里的实时库存。本节定义一个极简的 `get_stock` 工具，让程序可以根据商品名称读取当前库存。

In [2]:
inventory = {"机械键盘": 12, "无线鼠标": 7}

def get_stock(product_name):
    # 根据商品名称读取程序中的当前库存
    return inventory[product_name]

print("库存工具已就绪")

库存工具已就绪


输出说明库存工具已经定义，但此时还没有查询任何商品。下一步需要把这个普通 Python 函数写成大模型能够理解的工具说明。

## 2.2 说明工具的调用格式
大模型无法直接阅读 Python 函数。本节使用 JSON Schema 告诉大模型工具的名称、用途和必填参数，使它能够返回结构化的工具请求。

In [3]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_stock",
        "description": "查询指定商品的当前库存数量",
        "parameters": {
            "type": "object",
            "properties": {
                # product_name 是调用工具时需要填写的商品名称
                "product_name": {"type": "string"}
            },
            "required": ["product_name"],
        },
    },
}]

print(f"工具名称：{tools[0]['function']['name']}")
print(f"必填参数：{tools[0]['function']['parameters']['required']}")

工具名称：get_stock
必填参数：['product_name']


输出显示大模型将看到 `get_stock` 工具，并且调用时必须提供 `product_name`。工具说明已经准备好，下一步写出需要大模型完成的具体任务。

## 2.3 写出库存查询任务
为了确认答案确实来自外部工具，本节要求大模型查询程序中才有的库存数据，并明确告诉它不要猜测结果。

In [4]:
target_product = "无线鼠标"
messages = [
    {
        "role": "system",
        # 库存只能通过工具取得，不能让大模型自行猜测
        "content": "你是库存助手。查询库存时必须调用 get_stock，不要猜测结果。",
    },
    {
        "role": "user",
        "content": f"{target_product}还有多少库存？",
    },
]

print(f"任务：{messages[-1]['content']}")

任务：无线鼠标还有多少库存？


输出显示本次任务是查询无线鼠标库存。这个答案不存在于消息中，必须执行 `get_stock` 才能得到；下一步固定两条执行路径共同使用的成功标准。

## 2.4 定义成功标准
基线版本和改进版本必须用同一把尺子判断结果。本节把工具中的真实库存保存为标准答案：只有返回无线鼠标的实际库存数量，任务才算成功。

In [5]:
# 从任务数据中读取唯一的标准答案
expected_stock = inventory[target_product]
print(f"成功标准：{target_product}的库存必须是 {expected_stock}")

成功标准：无线鼠标的库存必须是 7


输出给出了唯一的正确结果。至此，工具、调用格式、任务和成功标准都已准备完成；下一章将把这些内容发送给真实大模型，并查看它返回的工具请求。

# 3. 获取并验证 API 响应
## 3.1 获取真实响应
工具和任务已经准备完成。本节把消息和工具说明一起发送给真实大模型，并要求它必须选择一个工具；同时记录等待响应所用的时间。

In [6]:
from time import perf_counter

# 记录真实 API 请求的等待时间
request_started = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",
    temperature=0,
)
api_latency_ms = round((perf_counter() - request_started) * 1000)
print("真实 API 响应已收到")

真实 API 响应已收到


输出说明真实大模型已经返回响应，但库存工具还没有执行。下一步查看响应中是普通文字，还是可以交给程序处理的工具请求。

## 3.2 查看响应类型
一次模型响应可能包含文字，也可能包含工具请求。本节分别显示这两部分，确认大模型是否按照工具说明选择了 `get_stock`。

In [7]:
# 取出大模型返回的完整消息
assistant_message = response.choices[0].message
print(f"文字内容：{assistant_message.content}")
print(f"工具请求数量：{len(assistant_message.tool_calls)}")

文字内容：我来帮您查询无线鼠标的库存数量。
工具请求数量：1


输出中的工具请求数量为 `1`，说明大模型返回了一条程序可以处理的结构化请求。说明文字只是给用户看的提示，真正需要执行的操作保存在工具请求中；下一步取出其中的关键数据。

## 3.3 保存工具请求
程序执行工具需要知道调用编号、工具名称和参数。本节从响应中取出这三项数据，并保留大模型返回的原始参数文本。

In [8]:
# 保存后续章节共同使用的工具请求
tool_call = assistant_message.tool_calls[0]
call_id = tool_call.id
tool_name = tool_call.function.name
arguments_text = tool_call.function.arguments

print(f"调用编号：{call_id}")
print(f"工具名称：{tool_name}")
print(f"原始参数：{arguments_text}")

调用编号：call_db8ce92c1030463ebd68d5fe
工具名称：get_stock
原始参数：{"product_name": "无线鼠标"}


输出显示大模型选择了 `get_stock`，参数中包含需要查询的商品名称。原始参数仍然是 JSON 文本，后续基线版本和改进版本将用不同方式处理同一份工具请求。

## 3.4 查看本次请求信息
工具请求已经保存，还需要知道它来自哪个模型、为什么停止，以及消耗了多少 Token。本节集中显示这些真实 API 运行信息。

In [9]:
# 读取真实响应附带的运行信息
choice = response.choices[0]
usage = response.usage

print(f"Provider：{config['NANO_BACKEND']}")
print(f"Model：{model_name}")
print(f"停止原因：{choice.finish_reason}")
print(f"输入 Token：{usage.prompt_tokens}")
print(f"输出 Token：{usage.completion_tokens}")
print(f"总 Token：{usage.total_tokens}")
print(f"等待时间：{api_latency_ms} ms")

Provider：openai
Model：LongCat-2.0
停止原因：tool_calls
输入 Token：161
输出 Token：74
总 Token：235
等待时间：2675 ms


输出记录了本次真实 API 请求的来源、停止原因、Token 和延迟。停止原因表示大模型正在等待外层程序执行工具，并不代表库存查询已经完成；下一章将定义只读取文字内容的基线组件。

# 4. 定义基线组件
## 4.1 定义文字读取器
普通聊天程序通常只读取大模型返回的文字，并把这段文字直接当成最终答案。本节把这种做法定义为基线组件；它看不到单独保存在 `tool_calls` 中的工具请求。

In [10]:
def read_text_only(message):
    # 基线组件只读取文字，不处理工具请求
    return message.content

print("基线组件：只读取文字内容")

基线组件：只读取文字内容


输出说明基线组件已经定义，但还没有处理第 3 章的真实响应。下一章会把同一条模型消息交给它，查看只读取文字能否得到真实库存。

# 5. 展示基线故障
## 5.1 只读取模型文字
第 3 章的真实响应同时包含文字和工具请求。本节把这条响应交给基线组件，让它只取出文字，并把文字直接当作任务答案。

In [11]:
# 基线组件只读取同一条真实模型消息的文字部分
baseline_answer = read_text_only(assistant_message)
print(f"基线答案：{baseline_answer}")

基线答案：我来帮您查询无线鼠标的库存数量。


输出只是大模型准备查询库存时给出的说明，并不是库存工具返回的数值。工具请求仍然留在响应中，没有被基线组件执行；下一步用共同标准判断任务是否完成。

## 5.2 判断基线结果
第 2 章已经规定，成功结果必须是工具中的实际库存数量。本节把基线答案与标准答案直接比较，得到明确的任务结果。

In [12]:
# 使用第 2 章固定的标准答案判断基线结果
baseline_success = baseline_answer == expected_stock
print(f"标准答案：{expected_stock}")
print(f"基线任务成功：{baseline_success}")

标准答案：7
基线任务成功：False


输出为 `False`。大模型已经正确选择了工具和参数，但基线组件只读取文字，导致工具没有执行，真实库存也没有返回。下一章将定义能够读取并执行结构化工具请求的改进组件。

# 6. 定义改进组件
## 6.1 建立工具注册表
大模型返回的是工具名称，程序需要用这个名称找到真正的 Python 函数。本节建立一个极简的工具注册表，把 `get_stock` 名称连接到第 2 章定义的库存函数。

In [13]:
# 工具注册表负责连接工具名称和真实函数
tool_functions = {"get_stock": get_stock}
print(f"已注册工具：{list(tool_functions)}")

已注册工具：['get_stock']


输出显示 `get_stock` 已经注册。程序现在可以通过模型返回的名称找到库存函数；下一步定义统一执行器，把完整工具请求转换成函数调用。

## 6.2 定义工具执行器
当前通用做法是让一个统一执行器处理结构化工具请求：先把 JSON 参数文本还原成数据，再从注册表找到函数并传入参数，最后生成带有原调用编号的 `tool` 消息。

In [14]:
import json

def execute_tool_call(tool_call):
    # 把参数文本还原成数据，再找到并执行对应函数
    arguments = json.loads(tool_call.function.arguments)
    tool_function = tool_functions[tool_call.function.name]
    tool_output = tool_function(**arguments)

    # 原调用编号让大模型知道这是谁的执行结果
    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": str(tool_output),
    }

print("改进组件：结构化工具执行器")

改进组件：结构化工具执行器


输出说明改进组件已经定义，但尚未执行第 3 章的工具请求。下一章会把同一条真实请求交给执行器，得到库存结果并保留调用编号。

# 7. 展示修复结果
## 7.1 执行同一条工具请求
基线版本忽略了第 3 章的工具请求。本节把完全相同的请求交给结构化工具执行器，让它读取参数并调用真正的 `get_stock` 函数。

In [15]:
# 执行第 3 章保存的同一条真实工具请求
tool_message = execute_tool_call(tool_call)
print(f"调用编号：{tool_message['tool_call_id']}")
print(f"工具结果：{tool_message['content']}")

调用编号：call_db8ce92c1030463ebd68d5fe
工具结果：7


输出中的工具结果是 `7`，并且回执保留了原调用编号，说明程序已经取得真实库存。下一步把这条回执放回对话，让大模型知道工具执行结果。

## 7.2 组装工具回执上下文
工具执行完成后，外层程序需要把大模型原来的工具请求和对应回执依次加入消息列表。本节只组装这两条消息，不再次调用模型。

In [16]:
# 保留原工具请求，再紧跟带相同调用编号的工具回执
assistant_tool_message = assistant_message.model_dump(exclude_none=True)
followup_messages = messages + [assistant_tool_message, tool_message]
print(f"消息数量：{len(followup_messages)}")
print(f"最后两条角色：{followup_messages[-2]['role']} -> {followup_messages[-1]['role']}")

消息数量：4
最后两条角色：assistant -> tool


输出显示消息顺序是 `assistant -> tool`：前一条提出调用，后一条返回结果。上下文已经完整，下一步把它发送给真实大模型生成最终答复。

## 7.3 获取最终答复
大模型只有收到工具回执，才能基于真实数据回答用户。本节发送完整消息列表，并要求这一步直接生成文字答复。

In [17]:
# 把工具回执发送给同一个真实模型
final_request_started = perf_counter()
final_response = client.chat.completions.create(
    model=model_name,
    messages=followup_messages,
    tools=tools,
    tool_choice="none",
    temperature=0,
)
final_latency_ms = round((perf_counter() - final_request_started) * 1000)
print("最终 API 响应已收到")

最终 API 响应已收到


输出说明真实大模型已经看到工具结果并返回最终响应。下一步查看它给用户的答案，以及这次请求的 Token、停止原因和等待时间。

## 7.4 查看最终答复
本节读取第二次真实 API 响应，集中显示最终文字和运行信息，确认大模型是在收到工具结果后正常结束。

In [18]:
# 读取最终答复及其真实运行信息
final_choice = final_response.choices[0]
final_answer = final_choice.message.content
final_usage = final_response.usage

print(f"最终答复：{final_answer}")
print(f"停止原因：{final_choice.finish_reason}")
print(f"输入 Token：{final_usage.prompt_tokens}")
print(f"输出 Token：{final_usage.completion_tokens}")
print(f"总 Token：{final_usage.total_tokens}")
print(f"等待时间：{final_latency_ms} ms")

最终答复：无线鼠标目前还有 **7** 件库存。
停止原因：stop
输入 Token：135
输出 Token：46
总 Token：181
等待时间：2762 ms


输出中的最终答复来自工具返回的库存数据，停止原因也从等待工具变成正常结束。下一步使用与基线版本相同的标准判断改进结果。

## 7.5 判断改进结果
展示文字仍不能代替任务判定。本节直接读取工具回执中的库存数量，并与第 2 章固定的标准答案比较。

In [19]:
# 使用与基线版本完全相同的标准答案判断结果
fixed_stock = int(tool_message["content"])
fixed_success = fixed_stock == expected_stock
print(f"标准答案：{expected_stock}")
print(f"改进任务成功：{fixed_success}")

标准答案：7
改进任务成功：True


输出为 `True`。模型、任务和首次响应都没有改变，唯一变化是外层程序读取并执行了结构化工具请求，再把关联回执交还给模型。下一章将汇总基线版本和改进版本的对照结果。

# 8. 汇总消融对照
## 8.1 对比两条执行路径
两条路径使用同一个模型、同一项任务和同一份首次 API 响应。基线版本停在模型文字，改进版本继续执行并回传工具结果；本节汇总它们的 API 次数、Token、等待时间和任务结果。

In [20]:
# 基线版本只包含第一次真实 API 请求
baseline_total_tokens = usage.total_tokens
baseline_total_latency = api_latency_ms

# 改进版本还包含工具回执后的第二次真实 API 请求
fixed_total_tokens = usage.total_tokens + final_usage.total_tokens
fixed_total_latency = api_latency_ms + final_latency_ms

print("版本 | 处理工具请求 | API 次数 | 总 Token | 总等待时间 | 任务成功")
print(f"基线版本 | 否 | 1 | {baseline_total_tokens} | {baseline_total_latency} ms | {baseline_success}")
print(f"改进版本 | 是 | 2 | {fixed_total_tokens} | {fixed_total_latency} ms | {fixed_success}")

版本 | 处理工具请求 | API 次数 | 总 Token | 总等待时间 | 任务成功
基线版本 | 否 | 1 | 235 | 2675 ms | False
改进版本 | 是 | 2 | 416 | 5437 ms | True


对照结果显示，基线版本虽然少一次 API 调用、Token 更少、等待时间更短，但没有完成任务；改进版本付出一次回传调用的成本，换来了正确结果。模型和首次响应完全相同，成功差异来自外层程序是否真正处理了结构化工具请求。

## 8.2 拓展

### nano 版省略了什么

nano 版只处理一次、单工具、参数正确的调用，没有覆盖并行调用、Schema 校验错误、工具异常、超时、取消、幂等性、重试、结果截断、流式回执和高风险确认。生产闭环还要把 call_id、权限决定和真实副作用关联到同一 trace，避免模型文字与实际工具结果混淆。

### 延伸阅读


1. 2026, [OpenAI, Function calling](https://developers.openai.com/api/docs/guides/function-calling)：结构化工具 Schema、调用结果与多轮响应闭环。
2. 2024, [StableToolBench](https://aclanthology.org/2024.findings-acl.664/)：稳定、可复现的大规模工具学习评测。
3. 2024, [tau-bench: A Benchmark for Tool-Agent-User Interaction](https://arxiv.org/abs/2406.12045)：工具调用必须同时满足领域规则与环境终态。